# Skin Cancer Classification with EfficientNet-B0 + Grad-CAM

This notebook trains a transfer learning model on the ISIC dataset and generates Grad-CAM explanations.

**Steps:**
1. Install dependencies
2. Download and organize ISIC dataset
3. Train EfficientNet-B0
4. Generate Grad-CAM visualizations
5. Single-image inference demo

## Step 0: Upload Kaggle API Token (First Time Only)

To download the ISIC dataset, you need a Kaggle account and API token:

1. Go to [kaggle.com](https://www.kaggle.com) → Account → API → Create New Token
2. Download `kaggle.json`
3. Run the cell below and upload `kaggle.json`

In [ ]:
from google.colab import files
import os

# Upload kaggle.json if you haven't already
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !mv kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle API configured!")
else:
    print("Kaggle API already configured")

## Step 1: Install Dependencies

In [ ]:
!pip install -q tensorflow>=2.13,<2.19 numpy pandas matplotlib seaborn scikit-learn pillow kagglehub
print("Dependencies installed!")

## Step 2: Download and Organize ISIC Dataset

This downloads the ~3GB ISIC dataset and organizes it into benign/malignant folders.

In [ ]:
!python setup_colab.py

## Step 3: Train EfficientNet-B0

This step:
- Builds EfficientNet-B0 with ImageNet weights
- Trains for up to 50 epochs (with early stopping)
- Saves the best model to `models/efficientnet_b0_best.keras`
- Generates confusion matrix and ROC curve in `results/metrics/`

**Estimated time:** 30-60 minutes on Colab T4 GPU

In [ ]:
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Train the model
!python model_efficientnet_b0.py

## Step 4: Generate Grad-CAM Visualizations

Samples 10 test images (5 benign, 5 malignant) and generates Grad-CAM heatmaps.

In [ ]:
!python xai_gradcam.py

# Display the grid of results
from IPython.display import Image, display
display(Image('results/gradcam/grid.png'))

## Step 5: Single-Image Inference Demo

Upload your own skin lesion image to test the model.

In [ ]:
# Upload an image for testing
from google.colab import files
import shutil

print("Upload a skin lesion image (jpg):")
uploaded = files.upload()
uploaded_path = list(uploaded.keys())[0]
shutil.move(uploaded_path, 'test_upload.jpg')
print(f"Saved as test_upload.jpg")

In [ ]:
# Run inference with Grad-CAM
!python inference.py --image test_upload.jpg

# Display result
from IPython.display import Image, display
display(Image('results/inference/test_upload_gradcam.png'))

## Download Results

Download the trained model and all outputs:

In [ ]:
# Zip all results
!zip -r skin_cancer_results.zip models/ results/

from google.colab import files
files.download('skin_cancer_results.zip')

## Optional: Test on a Sample from Dataset

If you don't have your own image, test on one from the test set:

In [ ]:
import glob
import random

# Get a random test image
test_images = glob.glob('data/test/*/*.jpg')
sample = random.choice(test_images)
print(f"Testing on: {sample}")

!python inference.py --image "{sample}"

from IPython.display import Image, display
display(Image(f"results/inference/{sample.split('/')[-1].replace('.jpg', '')}_gradcam.png"))